# Last.fm batch recommendations

Run `recommend_album()` over a subset of `albums.csv` and write results to `datasets/lastfm_recommendations_<subset>_<strategy>.csv`.

Each recommendation is one row with a `rank` column (same shape as `recommendations_album_level_avg_embeddings.parquet`).

Re-run the batch cell to resume — albums with `status` already set are skipped.

In [ ]:
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from lastfm_albums import recommend_album, set_request_delay

In [ ]:
DATA_DIR = Path("../datasets")
SOURCE_PATH = DATA_DIR / "albums.csv"

TRACK_SELECTION = "random"  # top_listener | random | top_n_tracks
TOP_N = 3           # number of seed tracks used by top_n_tracks strategy
N_RECS = 5          # recommendations to return per album
FETCH_FLOOR = 20    # similar tracks fetched per seed track (larger = more candidates, more API calls)
REQUEST_DELAY_SEC = .79   # seconds between API requests
RANDOM_SEED = 42    # seed for random track selection (strategy "random")
SAVE_EVERY = 10     # checkpoint to CSV every N albums

## Subset

Pick which albums to process. Output file name follows the subset key and `TRACK_SELECTION`.

In [ ]:
SUBSETS = {
    "all": lambda df: df,
    "3plus": lambda df: df.loc[df["review_count"] > 2],
    "test": lambda df: df.sample(250),
    "1k": lambda df: df.sample(1000),
}

In [ ]:
SUBSET = "1k" 
MAX_ALBUMS = None  # e.g. 10 for a dry run

In [ ]:
albums = SUBSETS[SUBSET](pd.read_csv(SOURCE_PATH))
if MAX_ALBUMS is not None:
    albums = albums.head(MAX_ALBUMS)

strategy_key = (
    f"top_n_tracks_{TOP_N}" if TRACK_SELECTION == "top_n_tracks" else TRACK_SELECTION
)
OUTPUT_PATH = DATA_DIR / f"lastfm_recommendations_{SUBSET}_{strategy_key}.csv"
print(f"Subset: {SUBSET} ({len(albums):,} albums)")
print(f"Strategy: {TRACK_SELECTION}")
print(f"Output: {OUTPUT_PATH}")

In [ ]:
OUTPUT_COLS = [
    "album_id", "artist", "album", "review_count",
    "strategy", "status", "error", "seed_track",
    "rec_artist", "rec_album", "score", "rank",
]

results = pd.read_csv(OUTPUT_PATH) if OUTPUT_PATH.exists() else pd.DataFrame(columns=OUTPUT_COLS)

results["strategy"] = results["strategy"].astype("string")
results["status"] = results["status"].astype("string")
results["error"] = results["error"].astype("string")
results["seed_track"] = results["seed_track"].astype("string")
results["rec_artist"] = results["rec_artist"].astype("string")
results["rec_album"] = results["rec_album"].astype("string")
results["score"] = pd.to_numeric(results["score"], errors="coerce")
results["rank"] = pd.to_numeric(results["rank"], errors="coerce").astype("Int64")

processed_ids = set(
    results.loc[
        results["status"].notna() & (results["status"].astype(str).str.strip() != ""),
        "album_id",
    ]
)
pending = albums[~albums["album_id"].isin(processed_ids)]

print(f"Strategy: {TRACK_SELECTION}")
print(f"Pending: {len(pending):,} / {len(albums):,}")
results.head(10)

In [ ]:
set_request_delay(REQUEST_DELAY_SEC)

new_rows: list[dict] = []
since_save = 0

for _, row in tqdm(pending.iterrows(), total=len(pending), desc="Last.fm batch"):
    album_seed = RANDOM_SEED + int(row["album_id"])
    base = {
        "album_id": row["album_id"],
        "artist": row["artist"],
        "album": row["album"],
        "review_count": row["review_count"],
        "strategy": TRACK_SELECTION,
    }

    try:
        _, seed_track, recs, _ = recommend_album(
            row["album"],
            artist=row["artist"],
            n_recs=N_RECS,
            fetch_floor=FETCH_FLOOR,
            track_selection=TRACK_SELECTION,
            top_n=TOP_N,
            random_seed=album_seed,
            clear_cache=False,
        )
        if recs.empty:
            new_rows.append({
                **base,
                "status": "no_results",
                "error": pd.NA,
                "seed_track": pd.NA,
                "rec_artist": pd.NA,
                "rec_album": pd.NA,
                "score": pd.NA,
                "rank": pd.NA,
            })
        else:
            for rank, (_, rec) in enumerate(recs.iterrows(), start=1):
                if rank > N_RECS:
                    break
                new_rows.append({
                    **base,
                    "status": "ok",
                    "error": pd.NA,
                    "seed_track": seed_track["name"] if seed_track else pd.NA,
                    "rec_artist": rec["artist"],
                    "rec_album": rec["album"],
                    "score": rec["similarity_score"],
                    "rank": rank,
                })
    except Exception as exc:
        new_rows.append({
            **base,
            "status": "error",
            "error": str(exc)[:500],
            "seed_track": pd.NA,
            "rec_artist": pd.NA,
            "rec_album": pd.NA,
            "score": pd.NA,
            "rank": pd.NA,
        })

    since_save += 1
    if since_save >= SAVE_EVERY:
        results = pd.concat([results, pd.DataFrame(new_rows)], ignore_index=True)
        new_rows = []
        results.to_csv(OUTPUT_PATH, index=False)
        since_save = 0

if new_rows:
    results = pd.concat([results, pd.DataFrame(new_rows)], ignore_index=True)
results.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")

In [ ]:
print(results["status"].value_counts(dropna=False))
results.loc[results["status"] == "ok", ["artist", "album", "seed_track", "rank", "rec_album", "rec_artist", "score"]].head(10)

In [ ]:
# # Reset failed albums, then re-run the batch cell
# retry_ids = results.loc[results["status"].isin(["error", "no_results"]), "album_id"].unique()
# results = results[~results["album_id"].isin(retry_ids)]
# pending = albums[~albums["album_id"].isin(
#     results.loc[results["status"].notna(), "album_id"].unique()
# )]
# print(f"Reset {len(retry_ids):,} albums; pending: {len(pending):,} — re-run batch cell")